# Train Model Notebook
Bu dosya veri setini yükler, önişleme yapar ve modelleri eğitir.

In [3]:
import tensorflow as tf
print(tf.__version__)
print(dir(tf.keras.preprocessing))

import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib 
import matplotlib.pyplot as plt
import seaborn as sns
import os

2.19.0
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'image', 'image_dataset_from_directory', 'sequence', 'text', 'text_dataset_from_directory', 'timeseries_dataset_from_array']


In [4]:


# Veri setini yükle
file_path = 'data/fake_reviews_dataset.csv'
dfall = pd.read_csv(file_path)
# Demoda 10-20 arası verimiz olsun dendiği için 40k olan verimi 20 ye düşünmek için frac sayısını değiştiriyorum
df = dfall.sample(frac=0.005, random_state=42)  
df['label'] = df['label'].map({'CG': 0, 'OR': 1})
df = df.rename(columns={'text_': 'review'})

In [5]:
# Temizleme ve öznitelik çıkarımı
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df['clean_review'] = df['review'].apply(clean_text)
df['review_length'] = df['review'].apply(lambda x: len(str(x)))
df['exclamation_count'] = df['review'].apply(lambda x: str(x).count('!'))
df['capital_word_ratio'] = df['review'].apply(lambda x: len([w for w in str(x).split() if w.isupper()]) / (len(str(x).split()) + 1e-5))

In [ ]:
# TF-IDF + Sayısal birleştirme
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=2000)
X_tfidf = tfidf.fit_transform(df['clean_review'])
X_numeric = df[['review_length', 'exclamation_count', 'capital_word_ratio']].values
X_combined = hstack([X_tfidf, X_numeric])
y = df['label'].values
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42, stratify=y)

In [7]:

# X_test ve y_test'i kaydet
joblib.dump(X_test, 'models/X_test_combined_demo.pkl')
joblib.dump(y_test, 'models/y_test_labels_demo.pkl')


['models/y_test_labels_demo.pkl']

In [8]:

# TF-IDF vektörleştiricisini kaydetme
joblib.dump(tfidf, 'models/tfidf_vectorizer_demo.pkl')


['models/tfidf_vectorizer_demo.pkl']

In [9]:
from sklearn.model_selection import GridSearchCV
import lightgbm as lgb

# LightGBM için parametre aralığı
param_grid = {
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100, 300],
    'max_depth': [5, 10],
    'num_leaves': [15, 31]
}

# GridSearch ile model optimizasyonu
lgb_model = lgb.LGBMClassifier(random_state=42)
grid_lgb = GridSearchCV(lgb_model, param_grid, cv=2, scoring='f1_macro', verbose=1, n_jobs=1)
grid_lgb.fit(X_train, y_train)

print("En iyi parametreler:", grid_lgb.best_params_)

# En iyi model
best_lgb_model = grid_lgb.best_estimator_
joblib.dump(best_lgb_model, "models/lightgbm_model_demo.pkl")


Fitting 2 folds for each of 16 candidates, totalling 32 fits
[LightGBM] [Info] Number of positive: 43, number of negative: 37
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000150 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 427
[LightGBM] [Info] Number of data points in the train set: 80, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.537500 -> initscore=0.150282
[LightGBM] [Info] Start training from score 0.150282
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warnin

C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature nam

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature nam

[LightGBM] [Info] Number of positive: 43, number of negative: 38
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000076 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 334
[LightGBM] [Info] Number of data points in the train set: 81, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.530864 -> initscore=0.123614
[LightGBM] [Info] Start training from score 0.123614
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Lig

C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


['models/lightgbm_model_demo.pkl']

In [10]:


# ==== LSTM TRAIN-ONLY (train/val/test split + güçlendirilmiş callbacks) ====
import os, numpy as np, tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import joblib

# --- Hazırlık ---
os.makedirs("models", exist_ok=True)
tf.random.set_seed(42)
np.random.seed(42)

# --- Tokenizer ---
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(df['clean_review'])

X_seq = tokenizer.texts_to_sequences(df['clean_review'])
X_pad = pad_sequences(X_seq, maxlen=200, padding='post', truncating='post')
y_lstm = df['label'].values

# --- Train / Validation / Test ayırma ---
# %70 Train, %15 Validation, %15 Test
X_train_lstm, X_temp_lstm, y_train_lstm, y_temp_lstm = train_test_split(
    X_pad, y_lstm, test_size=0.3, stratify=y_lstm, random_state=42
)
X_val_lstm, X_test_lstm, y_val_lstm, y_test_lstm = train_test_split(
    X_temp_lstm, y_temp_lstm, test_size=0.5, stratify=y_temp_lstm, random_state=42
)

# --- Model ---
model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=64, input_length=200))
model.add(LSTM(64, return_sequences=True, recurrent_dropout=0.1))
model.add(BatchNormalization()); model.add(Dropout(0.3))
model.add(LSTM(32, recurrent_dropout=0.1))
model.add(BatchNormalization()); model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# --- Callbacks ---
early = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)
checkpoint = ModelCheckpoint(
    "models/best_lstm_model_dataset1_demo.h5",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# --- TRAIN ---
history = model.fit(
    X_train_lstm, y_train_lstm,
    validation_data=(X_val_lstm, y_val_lstm),
    epochs=20,
    batch_size=128,
    callbacks=[early, reduce_lr, checkpoint],
    verbose=1
)

# --- Kayıt ---
model.save("models/lstm_model_dataset1_demo.h5")
joblib.dump(tokenizer, "models/lstm_tokenizer_dataset1_demo.pkl")
joblib.dump((X_test_lstm, y_test_lstm), "models/test_set_lstm_demo.pkl")

print("✅ Eğitim tamamlandı, model ve test seti kaydedildi.")



C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - accuracy: 0.4794 - loss: 1.0433
Epoch 1: val_loss improved from None to 0.69373, saving model to models/best_lstm_model_dataset1_demo.h5


2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 694ms/step - accuracy: 0.4823 - loss: 1.0273 - val_accuracy: 0.4000 - val_loss: 0.6937 - learning_rate: 0.0010
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.4940 - loss: 0.7946
Epoch 2: val_loss improved from 0.69373 to 0.69358, saving model to models/best_lstm_model_dataset1_demo.h5


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 235ms/step - accuracy: 0.5035 - loss: 0.7883 - val_accuracy: 0.5333 - val_loss: 0.6936 - learning_rate: 0.0010
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - accuracy: 0.5497 - loss: 0.8260
Epoch 3: val_loss did not improve from 0.69358
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step - accuracy: 0.5603 - loss: 0.8142 - val_accuracy: 0.5333 - val_loss: 0.6937 - learning_rate: 0.0010
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.5199 - loss: 0.8394 
Epoch 4: val_loss did not improve from 0.69358
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step - accuracy: 0.5319 - loss: 0.8257 - val_accuracy: 0.5333 - val_loss: 0.6939 - learning_rate: 0.0010
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.5277 - loss: 0.8019 
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 5: val_loss did not improve from 0.69358
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.5319 - loss: 0.7938 - val_accuracy: 0.5333 - val_loss: 0.694

✅ Eğitim tamamlandı, model ve test seti kaydedildi.


In [11]:

from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB

nb_params = {'alpha': [0.1, 0.5, 1.0]}
nb_grid = GridSearchCV(MultinomialNB(), nb_params, cv=3, scoring='f1_macro')
nb_grid.fit(X_train, y_train)
best_nb = nb_grid.best_estimator_
joblib.dump(best_nb, "models/naive_bayes_model_demo.pkl")


ValueError: 
All the 9 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
9 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\sklearn\naive_bayes.py", line 762, in fit
    self._count(X, Y)
  File "C:\Users\kayra\anaconda3\envs\tfenv\lib\site-packages\sklearn\naive_bayes.py", line 889, in _count
    check_non_negative(X, "MultinomialNB (input X)")
  File "C:\Users\kayra\anacond

In [ ]:

from sklearn.svm import LinearSVC

svm_params = {'C': [0.1, 1.0, 10.0]}
svm_grid = GridSearchCV(LinearSVC(), svm_params, cv=3, scoring='f1_macro')
svm_grid.fit(X_train, y_train)
best_svm = svm_grid.best_estimator_
joblib.dump(best_svm, "models/svm_model_demo.pkl")


In [ ]:

from sklearn.neural_network import MLPClassifier

mlp_params = {
    'hidden_layer_sizes': [(64,), (100,)],
    'alpha': [0.5, 1.0],
    'learning_rate': ['constant', 'adaptive']
}
mlp_grid = GridSearchCV(MLPClassifier(max_iter=300), mlp_params, cv=3, scoring='f1_macro')
mlp_grid.fit(X_train, y_train)
best_mlp = mlp_grid.best_estimator_
joblib.dump(best_mlp, "models/mlp_model_demo.pkl")


In [ ]:
# === REFERANS BÖLÜMLERİ ===

# GridSearch ile hiperparametre optimizasyonu:
# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

# TF-IDF vektörleştirici:
# https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

# Model kaydetme/yükleme için:
# https://scikit-learn.org/stable/modules/model_persistence.html#persistence-example

# MultinomialNB:
# https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html

# LinearSVC:
# https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html

# MLPClassifier:
# https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html

# LightGBMClassifier:
# https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMClassifier.html
